# Dataset Comparison: MODIS Snow Phenology v1 (Icechunk/Zarr v3) vs. Legacy (Zarr v2)

Compares the new `modis_snow_phenology_v1` dataset against the legacy `global_modis_snow_cover_4.zarr` tile by tile.

Checks:
- Global structure (shape, coordinates, dtypes, attributes)
- Per-tile value agreement (SAD_DOWY, SDD_DOWY, max_consec_snow_days)
- Fill value handling and valid pixel counts
- Coordinate precision differences

In [ ]:
import sys
from pathlib import Path

import adlfs
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

sys.path.insert(0, str(Path('..').resolve()))
from modis_snow_phenology.config import Config

config = Config('config/config_v1.txt')
FILL = np.iinfo(np.int16).min  # -32768
VARS = ['SAD_DOWY', 'SDD_DOWY', 'max_consec_snow_days']

## Open both stores

In [ ]:
# New store — Icechunk / Zarr v3
storage = icechunk.azure_storage(
    account=config.azure_storage_account,
    container=config.azure_container,
    prefix=config.icechunk_prefix,
    sas_token=config.azure_storage_sas_token,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session('main')
ds_new = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all')
print('New store:')
print(ds_new)

In [ ]:
# Legacy store — Zarr v2 on Azure Blob
# The old storage account may differ — adjust account_name if needed
old_sas = config.azure_storage_sas_token
old_fs = adlfs.AzureBlobFileSystem(account_name=config.azure_storage_account, credential=old_sas)
old_store = old_fs.get_mapper('snowmelt/snow_cover/global_modis_snow_cover_4.zarr')
ds_old = xr.open_zarr(old_store, consolidated=True)
print('Legacy store:')
print(ds_old)

## Global structure comparison

In [ ]:
print('=== Shape ===')
for v in VARS:
    s_new = ds_new[v].shape
    s_old = ds_old[v].shape
    match = '✅' if s_new == s_old else '❌'
    print(f'  {v}: new={s_new}  old={s_old}  {match}')

print()
print('=== Dtype ===')
for v in VARS:
    d_new = ds_new[v].dtype
    d_old = ds_old[v].dtype
    match = '✅' if d_new == d_old else '⚠️ '
    print(f'  {v}: new={d_new}  old={d_old}  {match}')

print()
print('=== Water years ===')
wy_new = ds_new.water_year.values
wy_old = ds_old.water_year.values
print(f'  new: {wy_new}')
print(f'  old: {wy_old}')
print(f'  match: {"✅" if np.array_equal(wy_new, wy_old) else "❌"}')

print()
print('=== Coordinate ranges ===')
for coord in ['y', 'x']:
    c_new = ds_new[coord].values
    c_old = ds_old[coord].values
    close = np.allclose(c_new, c_old, atol=1.0)
    max_diff = np.max(np.abs(c_new - c_old)) if len(c_new) == len(c_old) else 'length mismatch'
    print(f'  {coord}: len new={len(c_new)}, old={len(c_old)}, max_diff={max_diff:.4f}m  {"✅" if close else "❌"}')

print()
print('=== CRS ===')
try:
    import rioxarray  # noqa: F401
    crs_new = ds_new.rio.crs
    crs_old = ds_old.rio.crs
    print(f'  new: {crs_new}')
    print(f'  old: {crs_old}')
    print(f'  match: {"✅" if crs_new == crs_old else "⚠️ (check proj strings)"}')
except Exception as e:
    print(f'  CRS comparison error: {e}')

print()
print('=== Global attributes ===')
print('  new attrs:', dict(ds_new.attrs))
print('  old attrs:', dict(ds_old.attrs))

## Per-tile value comparison

For each tile, extract the `(water_year, 2400, 2400)` slab from both stores and compare.

In [ ]:
def compare_tile(h: int, v: int, var: str, wy: int | None = None) -> dict:
    """Compare one variable for a tile between new and old stores."""
    y_sl = slice(v * 2400, (v + 1) * 2400)
    x_sl = slice(h * 2400, (h + 1) * 2400)

    new_tile = ds_new[var].isel(y=y_sl, x=x_sl)
    old_tile = ds_old[var].isel(y=y_sl, x=x_sl)

    if wy is not None:
        new_tile = new_tile.sel(water_year=wy)
        old_tile = old_tile.sel(water_year=wy)

    n = new_tile.values.astype(np.int32)
    o = old_tile.values.astype(np.int32)

    n_valid = (n != FILL)
    o_valid = (o != FILL)
    both_valid = n_valid & o_valid

    result = {
        'tile': f'h{h:02d}v{v:02d}',
        'var': var,
        'wy': wy,
        'new_valid_pct': 100 * n_valid.mean(),
        'old_valid_pct': 100 * o_valid.mean(),
        'both_valid_n': int(both_valid.sum()),
    }

    if both_valid.any():
        diff = (n - o)[both_valid]
        result.update({
            'mean_diff': float(diff.mean()),
            'max_abs_diff': int(np.abs(diff).max()),
            'pct_exact_match': 100 * (diff == 0).mean(),
            'pct_within_1': 100 * (np.abs(diff) <= 1).mean(),
            'pct_within_8': 100 * (np.abs(diff) <= 8).mean(),  # within one 8-day composite
        })
    else:
        result.update({
            'mean_diff': np.nan, 'max_abs_diff': np.nan,
            'pct_exact_match': np.nan, 'pct_within_1': np.nan, 'pct_within_8': np.nan,
        })

    return result

In [ ]:
# Sample tiles across different regions — edit as needed
# Format: (h, v, label)
SAMPLE_TILES = [
    (10, 3, 'Sierra Nevada / California'),
    (11, 4, 'Rocky Mountains'),
    (19, 3, 'Alps / Europe'),
    (24, 5, 'Himalayas'),
    (10, 10, 'Patagonia (SH)'),
    (17, 2, 'Scandinavia'),
    (22, 4, 'Central Asia'),
    (28, 5, 'East Asia / Japan'),
]

SAMPLE_WYS = [2015, 2018, 2021, 2024]

print(f'Comparing {len(SAMPLE_TILES)} tiles × {len(SAMPLE_WYS)} water years × {len(VARS)} variables')
print(f'= {len(SAMPLE_TILES) * len(SAMPLE_WYS) * len(VARS)} comparisons')

In [ ]:
records = []

for h, v, label in SAMPLE_TILES:
    print(f'\n--- h{h:02d}v{v:02d} ({label}) ---')
    for wy in SAMPLE_WYS:
        for var in VARS:
            try:
                r = compare_tile(h, v, var, wy=wy)
                r['label'] = label
                records.append(r)
                print(f'  WY{wy} {var:25s}: '
                      f'exact={r["pct_exact_match"]:5.1f}%  '
                      f'within_8days={r["pct_within_8"]:5.1f}%  '
                      f'max_diff={r["max_abs_diff"]}  '
                      f'valid new={r["new_valid_pct"]:.1f}% old={r["old_valid_pct"]:.1f}%')
            except Exception as e:
                print(f'  WY{wy} {var}: ERROR — {e}')
                records.append({'tile': f'h{h:02d}v{v:02d}', 'var': var, 'wy': wy, 'label': label, 'error': str(e)})

df = pd.DataFrame(records)
print('\nDone.')

In [ ]:
print('=== Summary by variable ===')
print(df.groupby('var')[['pct_exact_match', 'pct_within_1', 'pct_within_8', 'max_abs_diff', 'mean_diff']].describe().round(2).to_string())

In [ ]:
print('=== Mean exact match % by tile and variable ===')
pivot = df.pivot_table(index='tile', columns='var', values='pct_exact_match', aggfunc='mean').round(1)
print(pivot.to_string())

## Visual comparison for one tile

Side-by-side maps of new vs. old for a single tile and water year.

In [ ]:
INSPECT_H, INSPECT_V = 10, 3   # Sierra Nevada
INSPECT_WY = 2021

y_sl = slice(INSPECT_V * 2400, (INSPECT_V + 1) * 2400)
x_sl = slice(INSPECT_H * 2400, (INSPECT_H + 1) * 2400)

fig, axes = plt.subplots(len(VARS), 3, figsize=(16, 4 * len(VARS)), dpi=120)
fig.suptitle(f'h{INSPECT_H:02d}v{INSPECT_V:02d}  WY{INSPECT_WY} — new vs. old vs. diff', fontsize=13)

for row, var in enumerate(VARS):
    new_arr = ds_new[var].sel(water_year=INSPECT_WY).isel(y=y_sl, x=x_sl).values.astype(float)
    old_arr = ds_old[var].sel(water_year=INSPECT_WY).isel(y=y_sl, x=x_sl).values.astype(float)

    new_arr[new_arr == FILL] = np.nan
    old_arr[old_arr == FILL] = np.nan
    diff = new_arr - old_arr

    vmin = np.nanpercentile(np.concatenate([new_arr.ravel(), old_arr.ravel()]), 2)
    vmax = np.nanpercentile(np.concatenate([new_arr.ravel(), old_arr.ravel()]), 98)
    abs_max_diff = np.nanpercentile(np.abs(diff), 99)

    axes[row, 0].imshow(new_arr, vmin=vmin, vmax=vmax, cmap='viridis', origin='upper')
    axes[row, 0].set_title(f'{var} — new')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(old_arr, vmin=vmin, vmax=vmax, cmap='viridis', origin='upper')
    axes[row, 1].set_title(f'{var} — old')
    axes[row, 1].axis('off')

    im = axes[row, 2].imshow(diff, vmin=-abs_max_diff, vmax=abs_max_diff, cmap='RdBu_r', origin='upper')
    axes[row, 2].set_title(f'{var} — diff (new − old)')
    axes[row, 2].axis('off')
    fig.colorbar(im, ax=axes[row, 2], shrink=0.8)

plt.tight_layout()
plt.show()

## Difference histograms

In [ ]:
fig, axes = plt.subplots(1, len(VARS), figsize=(15, 4), dpi=120)
fig.suptitle(f'h{INSPECT_H:02d}v{INSPECT_V:02d} WY{INSPECT_WY}: distribution of (new − old) where both valid')

y_sl = slice(INSPECT_V * 2400, (INSPECT_V + 1) * 2400)
x_sl = slice(INSPECT_H * 2400, (INSPECT_H + 1) * 2400)

for ax, var in zip(axes, VARS):
    n = ds_new[var].sel(water_year=INSPECT_WY).isel(y=y_sl, x=x_sl).values.astype(np.int32)
    o = ds_old[var].sel(water_year=INSPECT_WY).isel(y=y_sl, x=x_sl).values.astype(np.int32)
    both = (n != FILL) & (o != FILL)
    diff = (n - o)[both]

    ax.hist(diff, bins=50, color='steelblue', edgecolor='none')
    ax.axvline(0, color='red', lw=1.5)
    ax.set_title(var)
    ax.set_xlabel('new − old (days)')
    ax.set_ylabel('pixel count')
    ax.text(0.98, 0.95, f'n={len(diff):,}\nmean={diff.mean():.2f}\nstd={diff.std():.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8)

plt.tight_layout()
plt.show()

## Valid pixel count comparison across all sample tiles

In [ ]:
sub = df[df['var'] == 'SAD_DOWY'].dropna(subset=['new_valid_pct'])

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
x = np.arange(len(sub))
w = 0.35
ax.bar(x - w/2, sub['new_valid_pct'], w, label='new', color='steelblue')
ax.bar(x + w/2, sub['old_valid_pct'], w, label='old', color='orange')
ax.set_xticks(x)
ax.set_xticklabels([f"{r['tile']}\nWY{r['wy']}" for _, r in sub.iterrows()], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Valid pixel %')
ax.set_title('SAD_DOWY valid pixel count — new vs. old')
ax.legend()
plt.tight_layout()
plt.show()

## Variable attribute comparison

In [ ]:
for var in VARS:
    print(f'\n=== {var} ===')
    new_attrs = dict(ds_new[var].attrs)
    old_attrs = dict(ds_old[var].attrs)
    all_keys = sorted(set(new_attrs) | set(old_attrs))
    for k in all_keys:
        nv = new_attrs.get(k, '<missing>')
        ov = old_attrs.get(k, '<missing>')
        flag = '✅' if str(nv) == str(ov) else '⚠️ '
        print(f'  {flag} {k}: new={nv!r}  old={ov!r}')

## Zarr encoding comparison

In [ ]:
for var in VARS:
    print(f'\n=== {var} encoding ===')
    new_enc = ds_new[var].encoding
    old_enc = ds_old[var].encoding
    all_keys = sorted(set(new_enc) | set(old_enc))
    for k in all_keys:
        nv = new_enc.get(k, '<missing>')
        ov = old_enc.get(k, '<missing>')
        flag = '✅' if str(nv) == str(ov) else '⚠️ '
        print(f'  {flag} {k}: new={nv!r}  old={ov!r}')

## Final summary

In [ ]:
print('=== Overall comparison summary ===')
print(f'  Tiles compared    : {len(SAMPLE_TILES)}')
print(f'  Water years       : {SAMPLE_WYS}')
print(f'  Variables         : {VARS}')
print()

clean = df.dropna(subset=['pct_exact_match'])
for var in VARS:
    sub = clean[clean['var'] == var]
    print(f'  {var}:')
    print(f'    exact match  : {sub["pct_exact_match"].mean():.1f}% (mean across tiles/WYs)')
    print(f'    within 1 day : {sub["pct_within_1"].mean():.1f}%')
    print(f'    within 8 days: {sub["pct_within_8"].mean():.1f}%')
    print(f'    max abs diff : {sub["max_abs_diff"].max():.0f} days')

if 'error' in df.columns:
    errors = df[df['error'].notna()]
    if len(errors):
        print(f'\n  ⚠️  {len(errors)} comparisons failed with errors:')
        print(errors[['tile', 'var', 'wy', 'error']].to_string(index=False))